# PGD and adversarial training in the first codebase

The earliest attack line, built inside the `malware_detection` codebase (the 6,693/3,216-feature world of `01-data-and-features/`). Here PGD is a **class**, parameterised by norm (L∞, L2, L1), and it comes with its counterpart: **PGD adversarial training**. The notebook follows the codebase's own arc — PGD, then adversarial training, then the **Max** attack (run PGD in all norms, keep the best) and **StepwiseMax** — and closes with a small study of the inner-maximization step itself and the codebase's own test entry points.

**Where this fits:** Section 03 attacks the detectors built in `02-detectors/`, under the manipulation rules from `01-data-and-features/manipulation_maps.ipynb`. The defended models these attacks are later measured against live in `04-defenses/`, and the final, organised measurements are in `06-thesis-evaluation/`.

*Consolidated from `PGD2.ipynb` (the developed version, as the spine), `PGD.ipynb` (its earlier draft), `PGD_Max.ipynb`, `inner_maximizers.ipynb` and `examples_md_at_pgd_test.ipynb` (all originally in `a_malware`). The same PGD classes were also pasted into the old feature-importance notebook; that copy is not repeated here. Dataset-plumbing cells are in `01-data-and-features/building_the_dataset.ipynb`. Outputs cleared.*

## Step 1 — Setup: the codebase, the datasets, the model

In [ ]:
!pip install androguard
!pip install captum

import androguard
import captum

In [ ]:
import time
import os.path as path
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import os

torch.manual_seed(0)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

!git clone https://github.com/mostafa-ja/malware_detection.git

In [ ]:
import gdown
# our sharable address : https://drive.google.com/file/d/1-MPYbRTFn_zH3rybfAYihHuIjq2XSiI5/view?usp=sharing
download_link = 'https://drive.google.com/uc?id=1-MPYbRTFn_zH3rybfAYihHuIjq2XSiI5'
output_filepath = '/content/'
gdown.download(download_link, output_filepath)

In [ ]:
download_link = 'https://drive.google.com/uc?id=1--gjFTkAJUW74RXWE4anJ5wssV7K_VAo'
output_filepath = '/content/'
gdown.download(download_link, output_filepath)

In [ ]:
download_link = 'https://drive.google.com/uc?id=1-2NnBk3YKauGTGK8DfQHBxrLrYyTxNsp'
output_filepath = '/content/'
gdown.download(download_link, output_filepath)

In [ ]:
download_link = 'https://drive.google.com/uc?id=1-4PhQGYHpMXrCcxcgMadgPPA4S-9yluX'
output_filepath = '/content/'
gdown.download(download_link, output_filepath)

In [ ]:
!cp /content/train_dataset.pt /content/malware_detection/datasets
!cp /content/validation_dataset.pt /content/malware_detection/datasets
!cp /content/test_dataset.pt /content/malware_detection/datasets

In [ ]:
import torch
import torch.utils.data as data

class MyDataset(torch.utils.data.Dataset):
    def __init__(self, samples):
        self.samples = [(torch.tensor(data, dtype=torch.float32), label) for data, label in samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]

In [ ]:
class MalwareDetectionDNN(nn.Module):
    def __init__(self, input_size, n_classes, device='cpu', name='DNN', dense_hidden_units=[200, 200], dropout=0.6, smooth=False):
        """
        Construct malware detector

        Parameters
        ----------
        @param input_size: Integer, the dimentionality number of input vector
        @param n_classes: Integer, the number of classes, n=2
        @param device: String, 'cpu' or 'cuda'
        @param name: String, model name
        """
        super(MalwareDetectionDNN, self).__init__()
        self.input_size = input_size
        self.n_classes = n_classes
        self.name = name

        self.dropout = dropout
        self.smooth = smooth
        self.dense_hidden_units = dense_hidden_units

        self.dense_layers = []
        if len(self.dense_hidden_units) >= 1:
            self.dense_layers.append(nn.Linear(self.input_size, self.dense_hidden_units[0]))
        else:
            raise ValueError("Expect at least one hidden layer.")
        for i in range(len(self.dense_hidden_units[0:-1])):
            self.dense_layers.append(nn.Linear(self.dense_hidden_units[i],  # start from idx=1
                                               self.dense_hidden_units[i + 1]))
        self.dense_layers.append(nn.Linear(self.dense_hidden_units[-1], self.n_classes))

        # registration
        for idx_i, dense_layer in enumerate(self.dense_layers):
            self.add_module('nn_model_layer_{}'.format(idx_i), dense_layer)

        if self.smooth:
            self.activation_func = F.selu  # partial(F.elu, alpha=self.alpha_)
        else:
            self.activation_func = F.relu

        self.model_save_path = path.join('/content/malware_detection/save/drebin/md_dnn'+ '_' + self.name,'model.pth')

    def forward(self, x):
        """
        Go through the neural network

        Parameters
        ----------
        @param x: 2D tensor, feature representation
        """
        for dense_layer in self.dense_layers[:-1]:
            x = self.activation_func(dense_layer(x))

        latent_representation = F.dropout(x, self.dropout, training=self.training)
        logits = self.dense_layers[-1](latent_representation)
        return logits

    def customize_loss(self, logits, gt_labels, representation = None, mini_batch_idx=None):
        return F.cross_entropy(logits, gt_labels)

    # for predict pertubed malicious and create a y_true same size for theam
    def inference_batch_wise(self, x):
        """
        support malware samples solely
        """
        assert isinstance(x, torch.Tensor)
        logit = self.forward(x)
        return torch.softmax(logit, dim=-1).detach().cpu().numpy(), np.ones((logit.size()[0],))

    def inference(self, test_data_producer):
        confidences = []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer: # for large dataset we have to consider test dataset with batches
                x, y = x.double().to(device), y.long().to(device)
                logits = self.forward(x)
                confidences.append(F.softmax(logits, dim=-1))
                gt_labels.append(y)
        confidences = torch.vstack(confidences) #[[1,2,3],[4,5,6]] > below each other
        gt_labels = torch.cat(gt_labels, dim=0) #[[1,2,3],[4,5,6]] > [1,2,3,4,5,6]
        return confidences, gt_labels


    def predict(self, test_data_producer, indicator_masking=True):
        """
        predict labels and conduct evaluation

        Parameters
        --------
        @param test_data_producer, torch.DataLoader
        """
        # evaluation
        confidence, y_true = self.inference(test_data_producer)
        y_pred = confidence.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()
        from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
        accuracy = accuracy_score(y_true, y_pred)
        b_accuracy = balanced_accuracy_score(y_true, y_pred)
        print("The accuracy on the test dataset is {:.5f}%".format(accuracy * 100))
        print("The balanced accuracy on the test dataset is {:.5f}%".format(b_accuracy * 100))


        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        fpr = fp / float(tn + fp)
        fnr = fn / float(tp + fn)
        f1 = f1_score(y_true, y_pred, average='binary')

        print("Other evaluation metrics we may need:")
        print("False Negative Rate (FNR) is {:.5f}%, False Positive Rate (FPR) is {:.5f}%, F1 score is {:.5f}%".format(fnr * 100, fpr * 100, f1 * 100))

In [ ]:
%cd /content/malware_detection

## Step 2 — The feature vocabulary and which features are interdependent

In [ ]:
import pickle

# 1. Open the pickle file in binary read mode
with open('dataset/drebin/data.vocab_type', 'rb') as file:
    # 2. Use pickle.load to deserialize the data
    loaded_data1 = pickle.load(file)

# Now, loaded_data contains the data from the pickle file
print(loaded_data1)

# unique types in vocab
unique_types = set(loaded_data1)
print(unique_types)

In [ ]:
with open('dataset/drebin/data.vocab', 'rb') as file:
    loaded_data2 = pickle.load(file)

print(loaded_data2)

In [ ]:
for type in unique_types:
  filtered_data = [loaded_data2[index] for index, item in enumerate(loaded_data1) if item == type]
  print(type)
  print(filtered_data[:3])
  print()

In [ ]:
with open('dataset/drebin/data.vocab_info', 'rb') as file:
    loaded_data3 = pickle.load(file)

print(loaded_data3)

In [ ]:
for type in unique_types:
  filtered_data = [loaded_data3[index] for index, item in enumerate(loaded_data1) if item == type]
  print(type)
  print(filtered_data[:3])
  print()

In [ ]:
from core.droidfeature import InverseDroidFeature

inverse_feature = InverseDroidFeature()

# manipulation_x : removable features
manipulation_x = inverse_feature.get_manipulation()
manipulation_x = torch.LongTensor(manipulation_x).to(device)

# index of vocab which are interdependent
omega = inverse_feature.get_interdependent_apis()
omega = torch.sum(F.one_hot(torch.tensor(omega), num_classes=len(inverse_feature.vocab)),dim=0).to(device)

# a list of vocab length : if the type of feature is api , index of that feature will be true
api_flag = inverse_feature.get_api_flag()

In [ ]:
# index of vocab which are interdependent
omega = inverse_feature.get_interdependent_apis()
print(omega)

# [5,0,2] >> [1,0,1,0,0,1,0,0,0,0] length of vocab
# a list which interdependent feature are one
omega = torch.sum(F.one_hot(torch.tensor(omega), num_classes=len(inverse_feature.vocab)),dim=0).to(device)
print(omega)
print(omega[175])

In [ ]:
# a list of vocab length : if the type of feature is api , index of that feature will be true
api_flag = inverse_feature.get_api_flag()
api_flag[:10]

## Step 3 — The earlier draft's perturb() (from `PGD.ipynb`)

In [ ]:
import torch

from core.attack.base_attack import BaseAttack
from tools.utils import get_x0, round_x




EXP_OVER_FLOW = 1e-120



"""
Projected gradient descent (ascent).

Parameters
---------
@param norm, 'l2' or 'linf'
@param use_random, Boolean,  whether use random start point
@param rounding_threshold, float, a threshold for rounding real scalars
@param is_attacker, Boolean, play the role of attacker (note: the defender conducts adversarial training)
@param oblivion, Boolean, whether know the adversary indicator or not
@param kappa, attack confidence
@param manipulation_x, manipulations
@param omega, the indices of interdependent apis corresponding to each api
@param device, 'cpu' or 'cuda'
"""
norm = 'l2'
use_random = False
is_attacker = True
round_threshold = 0.5
lambda_ = 1.

def _perturb(model, x, label=None,steps=10,step_length=1.,round_threshold=0.5,lambda_=1):
    """
    perturb node feature vectors
    pertube a sample , and if loss not increased then adv_x stays the same x
    Parameters
    -----------
    @param model, a victim model
    @param x: torch.FloatTensor, node feature vectors (each represents the occurrences of apis in a graph) with shape [batch_size, vocab_dim]
    @param label: torch.LongTensor, ground truth labels
    @param steps: Integer, maximum number of iterations
    @param step_length: float, the step length in each iteration
    @param lambda_, float, penalty factor
    """

    adv_x = x
    loss_natural = 0.
    for t in range(steps):
        if t == 0 and use_random:
            adv_x = get_x0(adv_x, round_threshold, is_sample=True)
        var_adv_x = torch.autograd.Variable(adv_x, requires_grad=True)
        loss, done = get_loss(model, var_adv_x, label, lambda_)
        if t == 0:
            loss_natural = loss
        grad = torch.autograd.grad(torch.mean(loss), var_adv_x)[0].detach().data
        perturbation = get_perturbation(grad, x, adv_x)
        adv_x = torch.clamp(adv_x + perturbation * step_length, min=0., max=1.)
    # round
    if norm == 'linf' and (not hasattr(model, 'is_detector_enabled')):
        round_threshold = torch.rand(x.size()).to(device)
    else:
        round_threshold = round_threshold
    adv_x = round_x(adv_x, round_threshold)
    loss_adv, _1 = get_loss(model, adv_x, label, lambda_)
    replace_flag = (loss_adv < loss_natural).unsqueeze(1).expand_as(adv_x)
    adv_x[replace_flag] = x[replace_flag]
    return adv_x


def get_loss(model, adv_x, label, lambda_=None):

    logits_f = model(adv_x)

    ce = F.cross_entropy(logits_f, label, reduction='none')
    y_pred = logits_f.argmax(1)

    loss_no_reduction = ce
    done = y_pred != label
    return loss_no_reduction, done

def get_scores(model, pertb_x, label, lmda=1.):

    logits_f = model(pertb_x)
    y_pred = logits_f.argmax(1)

    done = y_pred != label
    return done



def get_perturbation(gradients, features, adv_features):
    # look for allowable position, because only '1--> -' and '0 --> +' are permitted
    # api insertion
    pos_insertion = (adv_features <= 0.5) * 1 * (adv_features >= 0.)
    grad4insertion = (gradients >= 0) * pos_insertion * gradients
    # grad4insertion = (gradients > 0) * gradients
    # api removal
    pos_removal = (adv_features > 0.5) * 1
    grad4removal = (gradients < 0) * (pos_removal & manipulation_x) * gradients
    if is_attacker:
        # cope with the interdependent apis
        checking_nonexist_api = (pos_removal ^ omega) & omega # ^ : XOR
        grad4removal[:, api_flag] += torch.sum(gradients * checking_nonexist_api, dim=-1, keepdim=True)
    gradients = grad4removal + grad4insertion

    # norm
    if norm == 'linf':
        perturbation = torch.sign(gradients)
    elif norm == 'l2':
        l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
        perturbation = torch.minimum(
            torch.tensor(1., dtype=features.dtype, device=features.device),
            gradients / l2norm
        )
        perturbation = torch.where(torch.isnan(perturbation), 0., perturbation)
        perturbation = torch.where(torch.isinf(perturbation), 1., perturbation)


    return perturbation

In [ ]:
def perturb(model, x, label=None,steps=10,step_length=1.,min_lambda_=1e-5,max_lambda_=1e5,base=10.):
    """
    enhance attack
    """

    lambda_ = max_lambda_

    # ensuring that the original data is not modified during the attack
    adv_x = x.detach().clone().to(torch.double)

    while lambda_ <= max_lambda_:
        with torch.no_grad():
            _, done = get_loss(model, adv_x, label, lambda_)
        # torch.all(input) : Tests if all elements in input evaluate to True
        if torch.all(done): # true if all labels of adv_x is different to x
            break
        # work on samples which their labels are not changed yet
        pert_x = _perturb(model, adv_x[~done], label[~done],steps,step_length,lambda_=lambda_)
        adv_x[~done] = pert_x
        lambda_ *= base # it means that in at max the loop will be ran just one time , more is for the model with detector_enabled
    with torch.no_grad():
        _, done = get_loss(model, adv_x, label, lambda_)
        print(f"pgd attack effectiveness {done.sum().item() / done.size()[0] * 100:.3f}%.")
    return adv_x

In [ ]:
print(samples.shape)
print(labels.shape)

In [ ]:
model = MalwareDetectionDNN(3216,2)

#IMPORTANT: model must be double means that its weights must be double for getting float results
model = model.double().to(device)
#IMPORTANT : we can use pretrain model to check how much pgd attack is effective against normal pretrained model
model.load_state_dict(torch.load('/content/malware_detection/MalwareDetectionDNN.pth'))
model.eval()

samples,labels = next(iter(test_Loader))
# consider this fact that model should be pretrained in normal samples
perturb(model, samples, labels.long(),steps=10,step_length=1.,min_lambda_=1e-5,max_lambda_=1e5,base=10.)

In [ ]:
# here we check if our x0 is not random start(use_random=False), how many features we should add to create adv sample from normal samples
# if we used random start , it could add more features
samples,labels = next(iter(test_Loader))
x,label = samples[0:1],labels[0:1]
print('label(must be malicious, 1) : ',label.item())
adv_x = perturb(model, x, label.long(),steps=10,step_length=1.,min_lambda_=1e-5,max_lambda_=1e5,base=10.)
print(torch.sum(x))
print(torch.sum(adv_x))
print('added feature : ', torch.sum(adv_x-x).item())  # we can see how many new features are added

## Step 4 — Rounding the start point: get_x0

In [ ]:
def get_x0(x, rounding_threshold=0.5, is_sample=False):
    """
    Helper function to randomly initialize the the inner maximizer algos
    randomize such that the functionality is preserved.
    :param x: training sample
    :param is_sample: flag to sample randomly from feasible area or return just x
    :return: randomly sampled feasible version of x
    """
    if is_sample:
        rand_x = round_x(torch.rand(x.size()), alpha=rounding_threshold)
        if x.is_cuda:
            rand_x = rand_x.cuda()
        return (rand_x.byte() | x.byte()).double()
    else:
        return x


def round_x(x, alpha=0.5):
    """
    rounds x by thresholding it according to alpha which can be a scalar or vector
    :param x:
    :param alpha: threshold parameter
    :return: a double tensor of 0s and 1s.
    """
    return (x >= alpha).double()

## Step 5 — The PGD class

# Projected gradient descent (ascent).

In [ ]:
class PGD():
    """
    Projected gradient descent (ascent).

    Parameters
    ---------
    @param norm, 'l2' or 'linf'
    @param use_random, Boolean,  whether use random start point
    @param rounding_threshold, float, a threshold for rounding real scalars
    @param is_attacker, Boolean, play the role of attacker (note: the defender conducts adversarial training)
    @param oblivion, Boolean, whether know the adversary indicator or not
    @param kappa, attack confidence
    @param manipulation_x, manipulations
    @param omega, the indices of interdependent apis corresponding to each api
    @param device, 'cpu' or 'cuda'
    """

    def __init__(self, norm, use_random=False, rounding_threshold=0.5,
                 is_attacker=True, manipulation_x=None, omega=None,api_flag=None, device=None):
        super(PGD, self).__init__()
        assert norm == 'l1' or norm == 'l2' or norm == 'linf', "Expect 'l1', 'l2' or 'linf'."
        self.norm = norm
        self.use_random = use_random
        self.round_threshold = rounding_threshold
        self.is_attacker = is_attacker
        self.lambda_ = 1.
        self.omeag = omega
        self.manipulation_x = manipulation_x
        self.api_flag = api_flag
        self.device = device
    def _perturb(self, model, x, label=None,
                 steps=10,
                 step_length=1.,
                 lambda_=1.,
                 ):
        """
        perturb node feature vectors

        Parameters
        -----------
        @param model, a victim model
        @param x: torch.FloatTensor, node feature vectors (each represents the occurrences of apis in a graph) with shape [batch_size, vocab_dim]
        @param label: torch.LongTensor, ground truth labels
        @param steps: Integer, maximum number of iterations
        @param step_length: float, the step length in each iteration
        @param lambda_, float, penalty factor
        """
        adv_x = x
        self.lambda_ = lambda_
        model.eval()
        loss_natural = 0.
        for t in range(steps):
            if t == 0 and self.use_random:
                adv_x = get_x0(adv_x, rounding_threshold=self.round_threshold, is_sample=True)
            var_adv_x = torch.autograd.Variable(adv_x, requires_grad=True)
            loss, done = self.get_loss(model, var_adv_x, label, self.lambda_)
            if t == 0:
                loss_natural = loss
            grad = torch.autograd.grad(torch.mean(loss), var_adv_x)[0].detach().data
            perturbation = self.get_perturbation(grad, x, adv_x)
            adv_x = torch.clamp(adv_x + perturbation * step_length, min=0., max=1.)
        # round
        if self.norm == 'linf' and (not hasattr(model, 'is_detector_enabled')):
            round_threshold = torch.rand(x.size()).to(self.device)
        else:
            round_threshold = self.round_threshold
        adv_x = round_x(adv_x, round_threshold)
        loss_adv, _1 = self.get_loss(model, adv_x, label, self.lambda_)
        replace_flag = (loss_adv < loss_natural).unsqueeze(1).expand_as(adv_x)
        adv_x[replace_flag] = x[replace_flag]
        return adv_x

    def perturb(self, model, x, label=None,
                steps=10,
                step_length=1.,
                min_lambda_=1e-5,
                max_lambda_=1e5,
                base=10.,
                verbose=False):
        """
        enhance attack
        """
        model.eval()
        if hasattr(model, 'is_detector_enabled'):
            self.lambda_ = min_lambda_
        else:
            self.lambda_ = max_lambda_

        adv_x = x.detach().clone().to(torch.double)
        while self.lambda_ <= max_lambda_:
            with torch.no_grad():
                _, done = self.get_loss(model, adv_x, label, self.lambda_)
            if torch.all(done):
                break
            pert_x = self._perturb(model, adv_x[~done], label[~done],
                                   steps,
                                   step_length,
                                   lambda_=self.lambda_
                                   )
            adv_x[~done] = pert_x
            self.lambda_ *= base
        with torch.no_grad():
            _, done = self.get_loss(model, adv_x, label, self.lambda_)
            print(f"pgd {self.norm}: attack effectiveness {done.sum().item() / done.size()[0] * 100:.3f}%.")
        return adv_x

    def get_perturbation(self, gradients, features, adv_features):
        # look for allowable position, because only '1--> -' and '0 --> +' are permitted
        # api insertion
        pos_insertion = (adv_features <= 0.5) * 1 * (adv_features >= 0.)
        grad4insertion = (gradients >= 0) * pos_insertion * gradients
        # grad4insertion = (gradients > 0) * gradients
        # api removal
        pos_removal = (adv_features > 0.5) * 1
        grad4removal = (gradients < 0) * (pos_removal & self.manipulation_x) * gradients
        if self.is_attacker:
            # cope with the interdependent apis
            checking_nonexist_api = (pos_removal ^ self.omega) & self.omega
            grad4removal[:, self.api_flag] += torch.sum(gradients * checking_nonexist_api, dim=-1, keepdim=True)
        gradients = grad4removal + grad4insertion

        # norm
        if self.norm == 'linf':
            perturbation = torch.sign(gradients)
        elif self.norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = torch.minimum(
                torch.tensor(1., dtype=features.dtype, device=features.device),
                gradients / l2norm
            )
            perturbation = torch.where(torch.isnan(perturbation), 0., perturbation)
            perturbation = torch.where(torch.isinf(perturbation), 1., perturbation)
        else:
            raise ValueError("Expect 'l2' or 'linf' norm.")

        # add the extra perturbation owing to the interdependent apis
        if self.norm == 'linf' and self.is_attacker:
            perturbation += torch.any(perturbation[:, self.api_flag] < 0, dim=-1,
                                      keepdim=True) * checking_nonexist_api
        if self.norm == 'l2' and self.is_attacker:
            min_val = torch.amin(perturbation, dim=-1, keepdim=True).clamp_(max=0.)
            perturbation += (torch.any(perturbation[:, self.api_flag] < 0, dim=-1,
                                       keepdim=True) * torch.abs(min_val) * checking_nonexist_api)
        return perturbation

    def get_loss(self, model, adv_x, label, lambda_=None):
        if hasattr(model, 'is_detector_enabled'):
            logits_f, prob_g = model.forward(adv_x)
        else:
            logits_f = model.forward(adv_x)

        ce = F.cross_entropy(logits_f, label, reduction='none')
        y_pred = logits_f.argmax(1)
        if hasattr(model, 'is_detector_enabled') and (not self.oblivion):
            assert lambda_ is not None
            tau = model.get_tau_sample_wise(y_pred)
            if self.is_attacker:
                loss_no_reduction = ce + lambda_ * (torch.clamp(tau - prob_g,
                                                                max=self.kappa)
                                                    )
            else:
                loss_no_reduction = ce + lambda_ * (tau - prob_g)
            done = (y_pred != label) & (prob_g <= tau)
        else:
            loss_no_reduction = ce
            done = y_pred != label
        return loss_no_reduction, done

## Step 6 — PGD adversarial training

In [ ]:
class PGDAdvTraining(object):
    """adversarial training incorporating pgd attack

    Parameters
    -------
    @param model, Object,  a model to be protected, e.g., MalwareDetector
    @attack_model: Object, adversary's model for generating adversarial malware on the feature space
    """

    def __init__(self, model, attack=None, attack_param=None):
        self.model = model
        self.attack = attack
        self.attack_param = attack_param

        self.name = self.model.name
        self.model_save_path =  path.join('/content/malware_detection/save/md_at_pgd' + '_' + 'train','model.pth')

    def fit(self, train_data_producer, validation_data_producer=None, adv_epochs=45,beta=0.001,lr=0.005,weight_decay=0, verbose=True):
        """
        Applying adversarial train to enhance the malware detector.

        Parameters
        -------
        @param train_data_producer: Object, an dataloader object for producing a batch of training data
        @param validation_data_producer: Object, an dataloader object for producing validation dataset
        @param epochs: Integer, epochs for adversarial training
        @param adv_epochs: Integer, epochs for adversarial training
        @param beta: Float, penalty factor for adversarial loss
        @param lr: Float, learning rate of Adam optimizer
        @param weight_decay: Float, penalty factor, default value 5e-4
        @param verbose: Boolean, whether to show verbose info
        """
        optimizer = optim.Adam(self.model.parameters(), lr=lr, weight_decay=weight_decay)
        total_time = 0.
        nbatches = len(train_data_producer)
        best_acc_val = 0.
        acc_val_adv_be = 0.
        best_epoch = 0
        for i in range(adv_epochs):
            losses, accuracies = [], []
            for idx_batch, (x_batch, y_batch) in enumerate(train_data_producer):
                x_batch, y_batch = x_batch.double().to(device), y_batch.long().to(device)
                batch_size = x_batch.shape[0]
                # make data
                mal_x_batch, ben_x_batch, mal_y_batch, ben_y_batch = x_batch[y_batch == 1], x_batch[y_batch == 0], y_batch[y_batch == 1], y_batch[y_batch == 0]
                start_time = time.time()
                self.model.eval()
                pertb_mal_x = self.attack.perturb(self.model, mal_x_batch, mal_y_batch,**self.attack_param)
                total_time += time.time() - start_time
                x_batch = torch.cat([ben_x_batch, pertb_mal_x], dim=0)
                y_batch = torch.cat([ben_y_batch, mal_y_batch])
                start_time = time.time()
                self.model.train()
                optimizer.zero_grad()
                logits = self.model.forward(x_batch)
                loss_train = self.model.customize_loss(logits,y_batch)

                loss_train.backward()
                optimizer.step()

                total_time += time.time() - start_time
                mins, secs = int(total_time / 60), int(total_time % 60)
                acc_train = (logits.argmax(1) == y_batch).sum().item()
                acc_train /= x_batch.size()[0]
                accuracies.append(acc_train)
                losses.append(loss_train.item())
                if verbose:
                    print(f'Mini batch: {i * nbatches + idx_batch + 1}/{adv_epochs * nbatches} | training time in {mins:.0f} minutes, {secs} seconds.')
                    print(f'Training loss (batch level): {losses[-1]:.4f} | Train accuracy: {acc_train * 100:.2f}%.')
            if verbose:
                print(f'Training loss (epoch level): {np.mean(losses):.4f} | Train accuracy: {np.mean(accuracies) * 100:.2f}')

            # select model
            self.model.eval()
            # long-time to train (save the model temporally in case of interruption)
            self.save_to_disk(i + 1, optimizer, self.model_save_path + '.tmp')

            res_val = []
            avg_acc_val = []
            for x_val, y_val in validation_data_producer:
                x_val, y_val = x_val.double().to(device), y_val.long().to(device)
                logits = self.model.forward(x_val)
                acc_val = (logits.argmax(1) == y_val).sum().item()
                acc_val /= x_val.size()[0]
                avg_acc_val.append(acc_val)

                mal_x_batch, mal_y_batch = x_val[y_val == 1], y_val[y_val == 1]
                pertb_mal_x = self.attack.perturb(self.model, mal_x_batch, mal_y_batch,**self.attack_param)
                y_cent_batch, x_density_batch = self.model.inference_batch_wise(pertb_mal_x)
                y_pred = np.argmax(y_cent_batch, axis=-1)
                res_val.append(y_pred == 1.)
            assert len(res_val) > 0
            res_val = np.concatenate(res_val)
            acc_val_adv = np.sum(res_val).astype(float) / res_val.shape[0]
            acc_val = (np.mean(avg_acc_val) + acc_val_adv) / 2.
            # Owing to we look for a new threshold after each epoch, this hinders the convergence of training.
            # We save the model's parameters at last several epochs as a well-trained model may be obtained.
            if acc_val >= best_acc_val:
                best_acc_val = acc_val
                acc_val_adv_be = acc_val_adv
                best_epoch = i + 1
                self.save_to_disk(best_epoch, optimizer, self.model_save_path)

            if verbose:
                print(f"\tVal accuracy {acc_val * 100:.4}% with accuracy {acc_val_adv * 100:.4}% under attack.")
                print(f"\tModel select at epoch {best_epoch} with validation accuracy {best_acc_val * 100:.4}% and accuracy {acc_val_adv_be * 100:.4}% under attack.")
                if hasattr(self.model, 'tau'):
                    print(f'The threshold is {self.model.tau}.')

    #we define here not in model definition, beacause we need attack.pertube()
    #also we must not use "with no grad" because we need grad for pertubation
    def adv_predict(self, test_data_producer):
      res_test = []
      for x, y in test_data_producer:
          x, y = x.double().to(device), y.long().to(device)

          mal_x_batch, mal_y_batch = x[y == 1], y[y == 1]
          pertb_mal_x = self.attack.perturb(self.model, mal_x_batch, mal_y_batch,**self.attack_param)
          y_cent_batch, x_density_batch = self.model.inference_batch_wise(pertb_mal_x)
          y_pred = np.argmax(y_cent_batch, axis=-1)
          res_test.append(y_pred == 1.)
      assert len(res_test) > 0
      res_test = np.concatenate(res_test)
      acc_val_adv = np.sum(res_test).astype(float) / res_test.shape[0]
      print(f"\tadversarial accuracy  {acc_val_adv * 100:.4}% under attack.")


    def load(self):
        assert path.exists(self.model_save_path), 'train model first'
        ckpt = torch.load(self.model_save_path)
        self.model.load_state_dict(ckpt['model'])

    def save_to_disk(self, epoch, optimizer, save_path=None):
        if not path.exists(save_path):
            os.makedirs(path.dirname(save_path), exist_ok = True)
        torch.save({'model': self.model.state_dict(),
                    'epoch': epoch,
                    'optimizer_state_dict': optimizer.state_dict()
                    },
                   save_path)

# Applying adversarial train to enhance the malware detector.

In [ ]:
train_dataset = torch.load('/content/malware_detection/datasets/train_dataset.pt')
train_Loader = data.DataLoader(train_dataset, batch_size=128, shuffle=False)
validation_dataset = torch.load('/content/malware_detection/datasets/validation_dataset.pt')
validation_Loader = data.DataLoader(validation_dataset, batch_size=10000, shuffle=False)
test_dataset = torch.load('/content/malware_detection/datasets/test_dataset.pt')
test_Loader = data.DataLoader(test_dataset, batch_size=10000, shuffle=False)

In [ ]:
import pickle
with open('/content/malware_detection/dataset/drebin/data.vocab', 'rb') as file:
    loaded_data = pickle.load(file)

print(len(loaded_data))

## Step 7 — Run it: L∞ adversarial training and the robustness check

In [ ]:
# norm = 'linf'

model = MalwareDetectionDNN(6693,2)
model = model.double().to(device)
attack = PGD(norm='linf', use_random=False,rounding_threshold=0.5,is_attacker=False,manipulation_x=manipulation_x, omega=omega,api_flag=api_flag,device=device)
attack_param = {}
max_adv_training_model = PGDAdvTraining(model, attack, attack_param)
max_adv_training_model.fit(train_Loader,validation_Loader,adv_epochs=30)

In [ ]:
# test: accuracy
max_adv_training_model.load()
max_adv_training_model.model.predict(test_Loader)

In [ ]:
max_adv_training_model.adv_predict(test_Loader)

## Step 8 — Run it: L2

In [ ]:
# norm = ''l2''

model = MalwareDetectionDNN(3216,2)
model = model.double().to(device)
attack = PGD(norm='l2', use_random=False,rounding_threshold=0.5,is_attacker=False,manipulation_x=manipulation_x, omega=omega,api_flag=api_flag,device=device)
attack_param = {}
max_adv_training_model = PGDAdvTraining(model, attack, attack_param)
max_adv_training_model.fit(train_Loader,validation_Loader,adv_epochs=30)

## Step 9 — The L1 variant: PGDl1

# Norm : L1

In [ ]:
class PGDl1():
    """
    Projected gradient descent (ascent) with gradients 'normalized' using l1 norm.
    By comparing BCA, the api removal is leveraged

    Parameters
    ---------
    @param is_attacker, Boolean, play the role of attacker (note: the defender conducts adversarial training)
    @param oblivion, Boolean, whether know the adversary indicator or not
    @param kappa, attack confidence
    @param manipulation_x, manipulations
    @param omega, the indices of interdependent apis corresponding to each api
    @param device, 'cpu' or 'cuda'
    """

    def __init__(self, is_attacker=True, oblivion=False, kappa=1., manipulation_x=None, omega=None, api_flag=None, device=None):
        super(PGDl1, self).__init__()
        self.is_attacker = is_attacker
        self.lambda_ = 1.
        self.omeag = omega
        self.manipulation_x = manipulation_x
        self.api_flag = api_flag
        self.device = device

    def _perturb(self, model, x,  label=None,
                 steps=10,
                 lambda_=1.):
        """
        perturb node feature vectors

        Parameters
        -----------
        @param model, a victim model
        @param x: torch.FloatTensor, node feature vectors (each represents the occurrences of apis in a graph) with shape [batch_size, number_of_graphs, vocab_dim]
        @param label: torch.LongTensor, ground truth labels
        @param steps: Integer, maximum number of perturbations
        @param lambda_, float, penalty factor
        """
        adv_x = x
        worst_x = x.detach().clone()
        self.lambda_ = lambda_
        self.padding_mask = torch.sum(adv_x, dim=-1, keepdim=True) > 1  # we set a graph contains two apis at least
        model.eval()
        for t in range(steps):
            var_adv_x = torch.autograd.Variable(adv_x, requires_grad=True)
            loss, done = self.get_loss(model, var_adv_x, label, self.lambda_)
            worst_x[done] = adv_x[done]
            if torch.all(done):
                break
            grad = torch.autograd.grad(torch.mean(loss), var_adv_x)[0]
            perturbation, direction = self.get_perturbation(grad, x, adv_x)
            # stop perturbing the examples that are successful to evade the victim
            perturbation[done] = 0.
            adv_x = torch.clamp(adv_x + perturbation * direction, min=0., max=1.)
        done = self.get_scores(model, adv_x, label)
        worst_x[done] = adv_x[done]
        return worst_x

    def perturb(self, model, x, label=None,
                steps=10,
                min_lambda_=1e-5,
                max_lambda_=1e5,
                base=10.,
                verbose=True):
        """
        enhance attack
        """
        model.eval()
        if hasattr(model, 'is_detector_enabled'):
            self.lambda_ = min_lambda_
        else:
            self.lambda_ = max_lambda_
        adv_x = x.detach().clone().to(torch.double)
        while self.lambda_ <= max_lambda_:
            _, done = self.get_loss(model, adv_x, label, self.lambda_)
            if torch.all(done):
                break
            pert_x = self._perturb(model, adv_x[~done], label[~done],
                                   steps,
                                   lambda_=self.lambda_
                                   )
            adv_x[~done] = pert_x
            self.lambda_ *= base
            #if not self.check_lambda(model):
                #break
        with torch.no_grad():
            _, done = self.get_loss(model, adv_x, label, self.lambda_)
            if verbose:
                print(f"pgd l1: attack effectiveness {done.sum().item() / x.size()[0]}.")
        return adv_x

    def get_perturbation(self, gradients, features, adv_features):
        # 1. mask paddings
        gradients = gradients * self.padding_mask

        # 2. look for allowable position, because only '1--> -' and '0 --> +' are permitted
        #    2.1 api insertion
        pos_insertion = (adv_features <= 0.5) * 1
        grad4insertion = (gradients > 0) * pos_insertion * gradients
        #    2.2 api removal
        pos_removal = (adv_features > 0.5) * 1
        grad4removal = (gradients <= 0) * (pos_removal & self.manipulation_x) * gradients
        if self.is_attacker:
            #     2.2.1 cope with the interdependent apis
            checking_nonexist_api = (pos_removal ^ self.omega) & self.omega
            grad4removal[:, self.api_flag] += torch.sum(gradients * checking_nonexist_api, dim=-1, keepdim=True)

        gradients = grad4removal + grad4insertion

        # 3. remove duplications
        un_mod = torch.abs(features - adv_features) <= 1e-6
        gradients = gradients * un_mod

        # 4. look for important position
        absolute_grad = torch.abs(gradients).reshape(features.shape[0], -1)
        _, position = torch.max(absolute_grad, dim=-1)
        perturbations = F.one_hot(position, num_classes=absolute_grad.shape[-1]).double()
        perturbations = perturbations.reshape(features.shape)
        directions = torch.sign(gradients) * (perturbations > 1e-6)

        # 5. tailor the interdependent apis
        if self.is_attacker:
            perturbations += (torch.any(directions[:, self.api_flag] < 0, dim=-1, keepdim=True)) * checking_nonexist_api
            directions += perturbations * self.omega
        return perturbations, directions


    def get_loss(self, model, adv_x, label, lambda_=None):
        if hasattr(model, 'is_detector_enabled'):
            logits_f, prob_g = model.forward(adv_x)
        else:
            logits_f = model.forward(adv_x)

        ce = F.cross_entropy(logits_f, label, reduction='none')
        y_pred = logits_f.argmax(1)
        if hasattr(model, 'is_detector_enabled') and (not self.oblivion):
            assert lambda_ is not None
            tau = model.get_tau_sample_wise(y_pred)
            if self.is_attacker:
                loss_no_reduction = ce + lambda_ * (torch.clamp(tau - prob_g,
                                                                max=self.kappa)
                                                    )
            else:
                loss_no_reduction = ce + lambda_ * (tau - prob_g)
            done = (y_pred != label) & (prob_g <= tau)
        else:
            loss_no_reduction = ce
            done = y_pred != label
        return loss_no_reduction, done

    def get_scores(self, model, pertb_x, label, lmda=1.):
        if hasattr(model, 'is_detector_enabled'):
            logits_f, prob_g = model.forward(pertb_x)
        else:
            logits_f = model.forward(pertb_x)
        y_pred = logits_f.argmax(1)
        if hasattr(model, 'is_detector_enabled') and (not self.oblivion):
            tau = model.get_tau_sample_wise(y_pred)
            done = (y_pred != label) & (prob_g <= tau)
        else:
            done = y_pred != label
        return done

In [ ]:
def __init__(self, is_attacker=True, oblivion=False, kappa=1., manipulation_x=None, omega=None, device=None):
        super(PGDl1, self).__init__(is_attacker, oblivion, kappa, manipulation_x, omega, device)

In [ ]:
model = MalwareDetectionDNN(3216,2)
model = model.double().to(device)
attack = PGDl1(is_attacker=False,manipulation_x=manipulation_x, omega=omega,api_flag=api_flag,device=device)
attack_param = {}
max_adv_training_model = PGDAdvTraining(model, attack, attack_param)
max_adv_training_model.fit(train_Loader,validation_Loader,adv_epochs=30)

## Step 10 — The Max attack: PGD in every norm, best result wins

In [ ]:
class Max():
    """
    max attack: select results from several attacks iteratively

    Parameters
    --------
    @param attack_list: List, a list of instantiated attack object
    @param varepsilon: Float, a scaler for justifying the convergence
    """
    def __init__(self, attack_list, varepsilon=1e-20,is_attacker=True, oblivion=False, kappa=1., manipulation_x=None, omega=None, device=None):
        super(Max, self).__init__()
        assert len(attack_list) > 0, 'Expect one attack at least.'
        self.attack_list = attack_list
        self.varepsilon = varepsilon
        self.device = device

    def perturb(self, model, x, label=None, steps_max=5, min_lambda_=1e-5, max_lambda_=1e5, verbose=False):
        """
        perturb node features

        Parameters
        -----------
        @param model, a victim model
        @param x: torch.FloatTensor, feature vectors with shape [batch_size, vocab_dim]
        @param label: torch.LongTensor, ground truth labels
        @param steps_max: Integer, maximum number of iterations
        @param min_lambda_: float, balance the importance of adversary detector, if it exists
        @param max_lambda_: float, balance the importance of adversary detector, if it exists
        @param verbose: Boolean, print verbose log
        """
        if x is None or x.shape[0] <= 0:
            return []
        model.eval()
        with torch.no_grad():
            loss, done = self.get_scores(model, x, label) #shape:[samples],[samples]
        pre_loss = loss
        n, red_n = x.size()[0], x.size()[1:] #samples,features
        red_ind = list(range(2, len(x.size()) + 1)) #2
        adv_x = x.detach().clone()
        stop_flag = torch.zeros(n, dtype=torch.bool, device=self.device) #[samples]
        for t in range(steps_max):
            num_sample_red = n - torch.sum(stop_flag)
            if num_sample_red <= 0:
                break

            red_label = label[~stop_flag]
            pertbx = []
            for attack in self.attack_list:
                assert 'perturb' in type(attack).__dict__.keys()
                if t > 0 and 'use_random' in attack.__dict__.keys():
                    attack.use_random = False
                if 'Orthogonal' in type(attack).__name__:
                    pertbx.append(attack.perturb(model=model, x=adv_x[~stop_flag], label=red_label))
                else:
                    pertbx.append(attack.perturb(model=model, x=adv_x[~stop_flag], label=red_label,
                                                 min_lambda_=1e-5,
                                                 max_lambda_=1e5,
                                                 ))
            # here pertbx.shape = a list of (number of attacks  ,(num_sample_red,features))
            pertbx = torch.vstack(pertbx)
            # here pertbx.shape = a tensor (num_sample_red*number of attacks samples, features)
            with torch.no_grad():
                red_label_ext = torch.cat([red_label] * len(self.attack_list)) #(labels*number of attacks )
                loss, done = self.get_scores(model, pertbx, red_label_ext) #(labels*number of attacks )
                loss = loss.reshape(len(self.attack_list), num_sample_red).permute(1, 0) #(num_sample_red,number of attacks) IMPORTANT: NOT(number of attacks, samples)
                done = done.reshape(len(self.attack_list), num_sample_red).permute(1, 0) #(num_sample_red,number of attacks)
                success_flag = torch.any(done, dim=-1) #(num_sample_red)
                # for a sample, if there is at least one successful attack, we will select the one with maximum loss;
                # while if no attacks evade the victim successful, all perturbed examples are reminded for selection
                done[~torch.any(done, dim=-1)] = 1 #loss.shape=done.shape=(samples,number of attacks)
                loss = (loss * done.to(torch.float)) + torch.min(loss) * (~done).to(torch.float) #(num_sample_red,number of attacks)
                pertbx = pertbx.reshape(len(self.attack_list), num_sample_red, *red_n).permute([1, 0, *red_ind])#(num_sample_red,attacks,features)
                _, indices = loss.max(dim=-1) # ans:(samples), max loss among attacks which worked, and max loss among all attacks for sample , none of them worked
                adv_x[~stop_flag] = pertbx[torch.arange(num_sample_red), indices]
                a_loss = loss[torch.arange(num_sample_red), indices]
                pre_stop_flag = stop_flag.clone()
                stop_flag[~stop_flag] = (torch.abs(pre_loss[~stop_flag] - a_loss) < self.varepsilon) | success_flag
                pre_loss[~pre_stop_flag] = a_loss
        if verbose:
            with torch.no_grad():
                _, done = self.get_scores(model, adv_x, label)
                print(f"max: attack effectiveness {done.sum().item() / x.size()[0] * 100}%.")
        return adv_x

    def get_scores(self, model, pertb_x, label):
        if hasattr(model, 'is_detector_enabled'):
            logits_f, prob_g = model.forward(pertb_x)
        else:
            logits_f = model.forward(pertb_x)
        ce = F.cross_entropy(logits_f, label, reduction='none')
        y_pred = logits_f.argmax(1)
        if hasattr(model, 'is_detector_enabled') and (not self.oblivion):
            tau = model.get_tau_sample_wise(y_pred)
            loss_no_reduction = -prob_g
            done = (y_pred != label) & (prob_g <= tau)
        else:
            loss_no_reduction = ce
            done = y_pred != label
        return loss_no_reduction, done

In [ ]:
attack = Max(attack_list=[pgdlinf, pgdl2, pgdl1],varepsilon=1e-9,is_attacker=False,device=device)
attack.perturb(model,x.to(device),label.to(device),steps_max=1)

In [ ]:
class MaxAdvTraining(object):
    """adversarial training incorporating mixture of attacks

    Parameters
    -------
    @param model, Object,  a model to be protected, e.g., MalwareDetector
    @attack_model: Object, adversary's model for generating adversarial malware on the feature space
    """

    def __init__(self, model, attack=None, attack_param=None):
        self.model = model
        if attack is not None:
            #assert isinstance(attack, (Max, StepwiseMax))
            if 'is_attacker' in attack.__dict__.keys():
                assert not attack.is_attacker
        self.attack = attack
        self.attack_param = attack_param

        self.name = self.model.name
        self.model_save_path =  path.join('/content/malware_detection/save/md_at_ma' + '_' + 'train','model.pth')

    def fit(self, train_data_producer, validation_data_producer=None, adv_epochs=50,
            beta=0.01,
            lr=0.001,
            under_sampling_ratio=1.,
            weight_decay=0, verbose=True):
        """
        Applying adversarial train to enhance the malware detector.

        Parameters
        -------
        @param train_data_producer: Object, an dataloader object for producing a batch of training data
        @param validation_data_producer: Object, an dataloader object for producing validation dataset
        @param epochs: Integer, epochs for adversarial training
        @param adv_epochs: Integer, epochs for adversarial training
        @param beta: Float, penalty factor for adversarial loss
        @param lr: Float, learning rate of Adam optimizer
        @param under_sampling_ratio: [0,1], under-sampling a portion of malware examples for adversarial training
        @param weight_decay: Float, penalty factor, default value 5e-4
        @param verbose: Boolean, whether to show verbose info
        """
        optimizer = optim.Adam(self.model.parameters(), lr=lr, weight_decay=weight_decay)
        total_time = 0.
        nbatches = len(train_data_producer)
        print("Max adversarial training is starting ...")
        best_acc_val = 0.
        acc_val_adv_be = 0.
        best_epoch = 0
        for i in range(adv_epochs):
            random.seed(0)
            losses, accuracies = [], []
            for idx_batch, (x_batch, y_batch) in enumerate(train_data_producer):
                x_batch, y_batch = x_batch.double().to(device), y_batch.long().to(device)
                batch_size = x_batch.shape[0]
                # make data
                mal_x_batch, ben_x_batch, mal_y_batch, ben_y_batch = x_batch[y_batch == 1], x_batch[y_batch == 0], y_batch[y_batch == 1], y_batch[y_batch == 0]
                if 0. < under_sampling_ratio < 1.:
                    n_mal = mal_x_batch.shape[0]
                    n_mal_sampling = int(under_sampling_ratio * n_mal) if int(under_sampling_ratio * n_mal) > 1 else 1
                    idx_sampling = random.sample(range(n_mal), n_mal_sampling)
                    mal_x_batch, mal_y_batch = mal_x_batch[idx_sampling], mal_y_batch[idx_sampling]

                start_time = time.time()
                self.model.eval()
                pertb_mal_x = self.attack.perturb(self.model, mal_x_batch, mal_y_batch,**self.attack_param)
                pertb_mal_x = round_x(pertb_mal_x, 0.5)
                total_time += time.time() - start_time
                x_batch = torch.cat([x_batch, pertb_mal_x], dim=0)
                y_batch = torch.cat([y_batch, mal_y_batch])
                start_time = time.time()
                self.model.train()
                optimizer.zero_grad()
                logits = self.model.forward(x_batch)
                loss_train = self.model.customize_loss(logits[:batch_size],y_batch[:batch_size])
                loss_train += beta * self.model.customize_loss(logits[batch_size:],y_batch[batch_size:])

                loss_train.backward()
                optimizer.step()
                total_time += time.time() - start_time
                mins, secs = int(total_time / 60), int(total_time % 60)
                acc_train = (logits.argmax(1) == y_batch).sum().item()
                acc_train /= x_batch.size()[0]
                accuracies.append(acc_train)
                losses.append(loss_train.item())
                if verbose:
                    print(f'Mini batch: {i * nbatches + idx_batch + 1}/{adv_epochs * nbatches} | training time in {mins:.0f} minutes, {secs} seconds.')
                    print(f'Training loss (batch level): {losses[-1]:.4f} | Train accuracy: {acc_train * 100:.2f}%.')
            if verbose:
                print(f'Training loss (epoch level): {np.mean(losses):.4f} | Train accuracy: {np.mean(accuracies) * 100:.2f}')

            # select model
            self.model.eval()
            # long-time to train (save the model temporally in case of interruption)
            self.save_to_disk(i + 1, optimizer, self.model_save_path + '.tmp')
            self.attack.is_attacker = True
            res_val = []
            avg_acc_val = []
            for x_val, y_val in validation_data_producer:
                x_val, y_val = x_val.double().to(device), y_val.long().to(device)
                logits = self.model.forward(x_val)
                acc_val = (logits.argmax(1) == y_val).sum().item()
                acc_val /= x_val.size()[0]
                avg_acc_val.append(acc_val)

                mal_x_batch, mal_y_batch = x_val[y_val == 1], y_val[y_val == 1]

                pertb_mal_x = self.attack.perturb(self.model, mal_x_batch, mal_y_batch,
                                                  **self.attack_param
                                                  )
                pertb_mal_x = round_x(pertb_mal_x, 0.5)
                y_cent_batch, x_density_batch = self.model.inference_batch_wise(pertb_mal_x)
                y_pred = np.argmax(y_cent_batch, axis=-1)
                res_val.append(y_pred == 1.)
            assert len(res_val) > 0
            res_val = np.concatenate(res_val)
            acc_val_adv = np.sum(res_val).astype(float) / res_val.shape[0]
            acc_val = (np.mean(avg_acc_val) + acc_val_adv) / 2.
            # Owing to we look for a new threshold after each epoch, this hinders the convergence of training.
            # We save the model's parameters at last several epochs as a well-trained model may be obtained.
            if acc_val >= best_acc_val:
                best_acc_val = acc_val
                acc_val_adv_be = acc_val_adv
                best_epoch = i + 1
                self.save_to_disk(best_epoch, optimizer, self.model_save_path)

            if verbose:
                print(f"\tVal accuracy {acc_val * 100:.4}% with accuracy {acc_val_adv * 100:.4}% under attack.")
                print(f"\tModel select at epoch {best_epoch} with validation accuracy {best_acc_val * 100:.4}% and accuracy {acc_val_adv_be * 100:.4}% under attack.")
                if hasattr(self.model, 'tau'):
                    print(
                        f'The threshold is {self.model.tau}.'
                    )
            self.attack.is_attacker = False

    def load(self):
        assert path.exists(self.model_save_path), 'train model first'
        ckpt = torch.load(self.model_save_path)
        # self.model.tau = ckpt['tau']
        # self.model.md_nn_model.load_state_dict(ckpt['md_model'])
        # self.model.load_state_dict(ckpt['amd_model'])
        self.model.load_state_dict(ckpt['model'])

    def save_to_disk(self, epoch, optimizer, save_path=None):
        if not path.exists(save_path):
            os.makedirs(path.dirname(save_path), exist_ok = True)
        # torch.save({'tau': self.model.tau,
        #             'md_model': self.model.md_nn_model.state_dict(),
        #             'amd_model': self.model.state_dict(),
        #             'epoch': epoch,
        #             'optimizer_state_dict': optimizer.state_dict()
        #             },
        #            save_path)
        torch.save({'model': self.model.state_dict(),
                    'epoch': epoch,
                    'optimizer_state_dict': optimizer.state_dict()
                    },
                   save_path)

## Step 11 — StepwiseMax in the first codebase

# StepwiseMax

In [ ]:
model = MalwareDetectionDNN(6693,2)
model = model.double().to(device)

In [ ]:
step_check=1
sl_l1=1.
sl_l2=1.
sl_linf=0.01
min_lambda_=1e-5
max_lambda_=1e5
is_score_round=True
base=10.
verbose=False
is_attacker = False

In [ ]:
def get_loss( model, adv_x, label, lambda_=None):
        if hasattr(model, 'is_detector_enabled'):
            logits_f, prob_g = model.forward(adv_x)
        else:
            logits_f = model.forward(adv_x)

        ce = F.cross_entropy(logits_f, label, reduction='none')
        y_pred = logits_f.argmax(1)
        if hasattr(model, 'is_detector_enabled') and (not oblivion):
            assert lambda_ is not None
            tau = model.get_tau_sample_wise(y_pred)
            if is_attacker:
                loss_no_reduction = ce + lambda_ * (torch.clamp(tau - prob_g,
                                                                max=kappa)
                                                    )
            else:
                loss_no_reduction = ce + lambda_ * (tau - prob_g)
            done = (y_pred != label) & (prob_g <= tau)
        else:
            loss_no_reduction = ce
            done = y_pred != label
        return loss_no_reduction, done

In [ ]:
def _perturb(model, x, label=None,
              steps=1,
              step_length_l1=1.,
              step_length_l2=0.5,
              step_length_linf=0.01,
              lambda_=1.,
              ):
    """
    perturb node feature vectors

    Parameters
    -----------
    @param model, a victim model
    @param x: torch.FloatTensor, node feature vectors (each represents the occurrences of apis in a graph) with shape [batch_size, vocab_dim]
    @param label: torch.LongTensor, ground truth labels
    @param steps: Integer, maximum number of iterations
    @param step_length_l1: float value in [0,1], the step length in each iteration
    @param step_length_l2: float, the step length in each iteration
    @param step_length_linf: float, the step length in each iteration
    @param lambda_, float, penalty factor
    """
    if x is None or x.shape[0] <= 0:
        return []
    lambda_ = lambda_
    assert 0 <= step_length_l1 <= 1, "Expected a real-value in [0,1], but got {}".format(step_length_l1)
    model.eval()
    adv_x = x.detach()

    def one_iteration(_adv_x, norm_type):
        var_adv_x = torch.autograd.Variable(_adv_x, requires_grad=True)
        loss, done = get_loss(model, var_adv_x, label, lambda_)
        grad = torch.autograd.grad(torch.mean(loss), var_adv_x)[0].data
        # look for allowable position, because only '1--> -' and '0 --> +' are permitted
        # api insertion
        pos_insertion = (_adv_x <= 0.5) * 1 * (_adv_x >= 0.)
        grad4insertion = (grad > 0) * pos_insertion * grad
        # api removal
        pos_removal = (_adv_x > 0.5) * 1
        grad4removal = (grad <= 0) * (pos_removal & manipulation_x) * grad
        if is_attacker:
            #     2.1 cope with the interdependent apis
            checking_nonexist_api = (pos_removal ^ omega) & omega
            grad4removal[:, api_flag] += torch.sum(grad * checking_nonexist_api, dim=-1, keepdim=True)
        grad = grad4removal + grad4insertion

        if norm_type == 'linf':
            perturbation = torch.sign(grad)
            if is_attacker:
                perturbation += (torch.any(perturbation[:, api_flag] < 0, dim=-1,
                                            keepdim=True) * checking_nonexist_api)
            return torch.clamp(_adv_x + step_length_linf * perturbation, min=0., max=1.)
        elif norm_type == 'l2':
            l2norm = torch.linalg.norm(grad, dim=-1, keepdim=True)
            perturbation = torch.minimum(
                torch.tensor(1., dtype=_adv_x.dtype, device=_adv_x.device),
                grad / l2norm
            )
            perturbation = torch.where(torch.isnan(perturbation), 0., perturbation)
            perturbation = torch.where(torch.isinf(perturbation), 1., perturbation)
            if is_attacker:
                min_val = torch.amin(perturbation, dim=-1, keepdim=True).clamp_(max=0.)
                perturbation += (torch.any(perturbation[:, api_flag] < 0, dim=-1,
                                            keepdim=True) * torch.abs(min_val) * checking_nonexist_api)
            return torch.clamp(_adv_x + step_length_l2 * perturbation, min=0., max=1.)
        elif norm_type == 'l1':
            # l1
            val, idx = torch.abs(grad).topk(int(1. / step_length_l1), dim=-1)
            perturbation = F.one_hot(idx, num_classes=_adv_x.shape[-1]).sum(dim=1)
            perturbation = torch.sign(grad) * perturbation
            if is_attacker:
                perturbation += (
                        torch.any(perturbation[:, api_flag] < 0, dim=-1,
                                  keepdim=True) * checking_nonexist_api)
            return torch.clamp(_adv_x + step_length_l1 * perturbation, min=0., max=1.)
        else:
            raise NotImplementedError

    adv_x_l1 = adv_x.clone()
    for t in range(steps):
        adv_x_l1 = one_iteration(adv_x_l1, norm_type='l1')
    adv_x_l2 = adv_x.clone()
    for t in range(steps):
        adv_x_l2 = one_iteration(adv_x_l2, norm_type='l2')
    adv_x_linf = adv_x.clone()
    for t in range(steps):
        adv_x_linf = one_iteration(adv_x_linf, norm_type='linf')
    return adv_x_l1, adv_x_l2, adv_x_linf

def get_scores(model, pertb_x, label):
    if hasattr(model, 'is_detector_enabled'):
        logits_f, prob_g = model.forward(pertb_x)
    else:
        logits_f = model.forward(pertb_x)
    y_pred = logits_f.argmax(1)
    ce = F.cross_entropy(logits_f, label, reduction='none')
    if hasattr(model, 'is_detector_enabled') and (not oblivion):
        tau = model.get_tau_sample_wise(y_pred)
        loss_no_reduction = ce - lambda_ * prob_g
        done = (y_pred != label) & (prob_g <= tau)
    else:
        loss_no_reduction = ce
        done = y_pred != label
    return loss_no_reduction, done

In [ ]:
class StepwiseMax():
    """
    Stepwise max attack (mixture of pgd l1, pgd l2, pgd linf)

    Parameters
    ---------
    @param use_random, Boolean,  whether use random start point
    @param rounding_threshold, float, a threshold for rounding real scalars
    @param is_attacker, Boolean, play the role of attacker (note: the defender conducts adversarial training)
    @param oblivion, Boolean, whether know the adversary indicator or not
    @param kappa, attack confidence
    @param manipulation_x, manipulations
    @param omega, the indices of interdependent apis corresponding to each api
    @param device, 'cpu' or 'cuda'
    """

    def __init__(self, use_random=False, rounding_threshold=0.5,
                 is_attacker=True, oblivion=False, kappa=1., manipulation_x=None, omega=None, device=None):
        super(StepwiseMax, self).__init__()
        self.use_random = use_random
        assert 0 < rounding_threshold < 1
        self.round_threshold = rounding_threshold
        self.lambda_ = 1.
        self.is_attacker = is_attacker

    def perturb(self, model, x, label=None,
                steps=100,
                step_check=1,
                sl_l1=1.,
                sl_l2=1.,
                sl_linf=0.01,
                min_lambda_=1e-5,
                max_lambda_=1e5,
                is_score_round=True,
                base=10.,
                verbose=False):
        """
        enhance attack
        """
        assert 0 < min_lambda_ <= max_lambda_
        assert steps >= 0 and (step_check >= 1) and 1 >= sl_l1 > 0 and sl_l2 >= 0 and sl_linf >= 0
        model.eval()
        if hasattr(model, 'is_detector_enabled'):
            self.lambda_ = min_lambda_
        else:
            self.lambda_ = max_lambda_
        if not self.is_attacker:
            # this can relieve the issue of setting a constant step length
            step_checks = [1, 10, 25, 50]
            step_check = random.choice(step_checks)
        mini_steps = [step_check] * (steps // step_check)
        mini_steps = mini_steps + [steps % step_check] if steps % step_check != 0 else mini_steps
        n, red_n = x.size()[0], x.size()[1:]
        red_ind = list(range(2, len(x.size()) + 1))

        adv_x = x.detach().clone()
        while self.lambda_ <= max_lambda_:
            pert_x_cont = None
            prev_done = None
            for i, mini_step in enumerate(mini_steps):
                with torch.no_grad():
                    if i == 0 and self.use_random:
                        adv_x = get_x0(adv_x, rounding_threshold=self.round_threshold, is_sample=True)
                    _, done = get_loss(model, adv_x, label, self.lambda_)
                if torch.all(done):
                    break
                if i == 0:
                    adv_x[~done] = x[~done]  # recompute the perturbation under other penalty factors
                    prev_done = done
                else:
                    adv_x[~done] = pert_x_cont[~done[~prev_done]]
                    prev_done = done

                num_sample_red = torch.sum(~done).item()
                pert_x_l1, pert_x_l2, pert_x_linf = self._perturb(model, adv_x[~done], label[~done],
                                                                  mini_step,
                                                                  sl_l1,
                                                                  sl_l2,
                                                                  sl_linf,
                                                                  lambda_=self.lambda_
                                                                  )
                with torch.no_grad():
                    pertb_x_list = [pert_x_linf, pert_x_l2, pert_x_l1]
                    n_attacks = len(pertb_x_list)
                    pertbx = torch.vstack(pertb_x_list)
                    label_ext = torch.cat([label[~done]] * n_attacks)
                    if (not self.is_attacker) and (not is_score_round):
                        scores, _done = self.get_scores(model, pertbx, label_ext)
                    else:
                        scores, _done = self.get_scores(model, round_x(pertbx, self.round_threshold), label_ext)
                    max_v = scores.amax() if scores.amax() > 0 else 0.
                    scores[_done] += max_v
                    pertbx = pertbx.reshape(n_attacks, num_sample_red, *red_n).permute([1, 0, *red_ind])
                    scores = scores.reshape(n_attacks, num_sample_red).permute(1, 0)
                    _2, s_idx = scores.max(dim=-1)
                    pert_x_cont = pertbx[torch.arange(num_sample_red), s_idx]
                    adv_x[~done] = pert_x_cont if not self.is_attacker else round_x(pert_x_cont, self.round_threshold)
            self.lambda_ *= base

        if self.is_attacker:
            adv_x = round_x(adv_x, self.round_threshold)
        with torch.no_grad():
            _, done = get_loss(model, adv_x, label, self.lambda_)
            if verbose:
                print(f"step-wise max: attack effectiveness {done.sum().item() / done.size()[0] * 100:.3f}%.")
        return adv_x

    def _perturb(self, model, x, label=None,
                 steps=1,
                 step_length_l1=1.,
                 step_length_l2=0.5,
                 step_length_linf=0.01,
                 lambda_=1.,
                 ):
        """
        perturb node feature vectors

        Parameters
        -----------
        @param model, a victim model
        @param x: torch.FloatTensor, node feature vectors (each represents the occurrences of apis in a graph) with shape [batch_size, vocab_dim]
        @param label: torch.LongTensor, ground truth labels
        @param steps: Integer, maximum number of iterations
        @param step_length_l1: float value in [0,1], the step length in each iteration
        @param step_length_l2: float, the step length in each iteration
        @param step_length_linf: float, the step length in each iteration
        @param lambda_, float, penalty factor
        """
        if x is None or x.shape[0] <= 0:
            return []
        self.lambda_ = lambda_
        assert 0 <= step_length_l1 <= 1, "Expected a real-value in [0,1], but got {}".format(step_length_l1)
        model.eval()
        adv_x = x.detach()

        def one_iteration(_adv_x, norm_type):
            var_adv_x = torch.autograd.Variable(_adv_x, requires_grad=True)
            loss, done = get_loss(model, var_adv_x, label, self.lambda_)
            grad = torch.autograd.grad(torch.mean(loss), var_adv_x)[0].data
            # look for allowable position, because only '1--> -' and '0 --> +' are permitted
            # api insertion
            pos_insertion = (_adv_x <= 0.5) * 1 * (_adv_x >= 0.)
            grad4insertion = (grad > 0) * pos_insertion * grad
            # api removal
            pos_removal = (_adv_x > 0.5) * 1
            grad4removal = (grad <= 0) * (pos_removal & manipulation_x) * grad
            if self.is_attacker:
                #     2.1 cope with the interdependent apis
                checking_nonexist_api = (pos_removal ^ omega) & omega
                grad4removal[:, api_flag] += torch.sum(grad * checking_nonexist_api, dim=-1, keepdim=True)
            grad = grad4removal + grad4insertion

            if norm_type == 'linf':
                perturbation = torch.sign(grad)
                if self.is_attacker:
                    perturbation += (torch.any(perturbation[:, api_flag] < 0, dim=-1,
                                               keepdim=True) * checking_nonexist_api)
                return torch.clamp(_adv_x + step_length_linf * perturbation, min=0., max=1.)
            elif norm_type == 'l2':
                l2norm = torch.linalg.norm(grad, dim=-1, keepdim=True)
                perturbation = torch.minimum(
                    torch.tensor(1., dtype=_adv_x.dtype, device=_adv_x.device),
                    grad / l2norm
                )
                perturbation = torch.where(torch.isnan(perturbation), 0., perturbation)
                perturbation = torch.where(torch.isinf(perturbation), 1., perturbation)
                if self.is_attacker:
                    min_val = torch.amin(perturbation, dim=-1, keepdim=True).clamp_(max=0.)
                    perturbation += (torch.any(perturbation[:, api_flag] < 0, dim=-1,
                                               keepdim=True) * torch.abs(min_val) * checking_nonexist_api)
                return torch.clamp(_adv_x + step_length_l2 * perturbation, min=0., max=1.)
            elif norm_type == 'l1':
                # l1
                val, idx = torch.abs(grad).topk(int(1. / step_length_l1), dim=-1)
                perturbation = F.one_hot(idx, num_classes=_adv_x.shape[-1]).sum(dim=1)
                perturbation = torch.sign(grad) * perturbation
                if self.is_attacker:
                    perturbation += (
                            torch.any(perturbation[:, api_flag] < 0, dim=-1,
                                      keepdim=True) * checking_nonexist_api)
                return torch.clamp(_adv_x + step_length_l1 * perturbation, min=0., max=1.)
            else:
                raise NotImplementedError

        adv_x_l1 = adv_x.clone()
        for t in range(steps):
            adv_x_l1 = one_iteration(adv_x_l1, norm_type='l1')
        adv_x_l2 = adv_x.clone()
        for t in range(steps):
            adv_x_l2 = one_iteration(adv_x_l2, norm_type='l2')
        adv_x_linf = adv_x.clone()
        for t in range(steps):
            adv_x_linf = one_iteration(adv_x_linf, norm_type='linf')
        return adv_x_l1, adv_x_l2, adv_x_linf

    def get_scores(self, model, pertb_x, label):
        if hasattr(model, 'is_detector_enabled'):
            logits_f, prob_g = model.forward(pertb_x)
        else:
            logits_f = model.forward(pertb_x)
        y_pred = logits_f.argmax(1)
        ce = F.cross_entropy(logits_f, label, reduction='none')
        if hasattr(model, 'is_detector_enabled') and (not self.oblivion):
            tau = model.get_tau_sample_wise(y_pred)
            loss_no_reduction = ce - self.lambda_ * prob_g
            done = (y_pred != label) & (prob_g <= tau)
        else:
            loss_no_reduction = ce
            done = y_pred != label
        return loss_no_reduction, done

In [ ]:
attack = StepwiseMax(is_attacker=False, device=device)
attack_param = {
    'steps': max(max(steps_l1, steps_linf), steps_l2),
    'sl_l1': 1.,
    'sl_l2': step_length_l2,
    'sl_linf': step_length_linf,
    'verbose': True
}

In [ ]:
max_adv_training_model = MaxAdvTraining(model, attack, attack_param)

max_adv_training_model.fit(train_Loader,
                          validation_Loader,
                          adv_epochs=50,
                          beta=0.01,
                          lr=0.001,
                          under_sampling_ratio=1.,
                          weight_decay=0.
                          )

## Step 12 — Study: the inner maximization, step by step

In [ ]:
def or_float_tensors(x_1, x_2):
    """
    ORs two float tensors by converting them to byte and back
    Note that byte() takes the first 8 bit after the decimal point of the float
    e.g., 0.0 ==> 0
          0.1 ==> 0
          1.1 ==> 1
        255.1 ==> 255
        256.1 ==> 0
    Subsequently the purpose of this function is to map 1s float tensors to 1
    and those of 0s to 0. I.e., it is meant to be used on tensors of 0s and 1s.

    :param x_1: tensor one
    :param x_2: tensor two
    :return: float tensor of 0s and 1s.
    """
    return (x_1.byte() | x_2.byte()).float()


def xor_float_tensors(x_1, x_2):
    """
    XORs two float tensors by converting them to byte and back
    Note that byte() takes the first 8 bit after the decimal point of the float
    e.g., 0.0 ==> 0
          0.1 ==> 0
          1.1 ==> 1
        255.1 ==> 255
        256.1 ==> 0
    Subsequently the purpose of this function is to map 1s float tensors to 1
    and those of 0s to 0. I.e., it is meant to be used on tensors of 0s and 1s.

    :param x_1: tensor one
    :param x_2: tensor two
    :return: float tensor of 0s and 1s.
    """
    return (x_1.byte() ^ x_2.byte()).float()


def clip_tensor(x, lb=0., ub=1.):
    """
    Clip a tensor to be within lb and ub
    :param x:
    :param lb: lower bound (scalar)
    :param ub: upper bound (scalar)
    :return: clipped version of x
    """
    return torch.clamp(x, min=lb, max=ub)

def round_x(x, alpha=0.99):
    """
    rounds x by thresholding it according to alpha which can be a scalar or vector
    :param x:
    :param alpha: threshold parameter
    :return: a float tensor of 0s and 1s.
    """
    return (x > alpha).float()


def get_x0(x, is_sample=False):
    """
    Helper function to randomly initialize the the inner maximizer algos
    randomize such that the functionality is preserved.
    Functionality is preserved by maintaining the features present in x
    :param x: training sample
    :param is_sample: flag to sample randomly from feasible area or return just x
    :return: randomly sampled feasible version of x
    """
    if is_sample:
        rand_x = round_x(torch.rand(x.size()))
        if x.is_cuda:
            rand_x = rand_x.cuda()
        return or_float_tensors(x, rand_x)
    else:
        return x

In [ ]:
samples,labels = next(iter(test_Loader))
x,y = samples[0:1],labels[0:1]
print(x.shape)
print(y.shape)

In [ ]:
# Convert samples and labels to PyTorch tensors and specify dtype as torch.float
# Convert samples and labels to a floating-point data type
x = x.clone().to(torch.float).requires_grad_(True)
y = y.clone().to(torch.long)

In [ ]:
loss_fct = torch.nn.CrossEntropyLoss()

# compute natural loss
loss_natural = loss_fct(model(x), y).data
loss_natural

In [ ]:
x_next = get_x0(x, is_sample=True)
print(torch.sum(x))
print(torch.sum(x_next))
# having alpha big number alpha(here 0.99) to decrease the diffrence between x and x_next

In [ ]:
# multi-step with gradients
k=10
epsilon = 0.02
for t in range(k):
    # forward pass
    x_var = x_next.requires_grad_()
    y_model = model(x_var)
    loss = loss_fct(y_model, y)
    print(loss)
    # compute gradient
    #grad_vars gives us the list of gradients of variables which are requires_grad compare to loss
    # here we just have one variable so in grad_vars there is just one element
    grad_vars = torch.autograd.grad(loss.mean(), x_var)


    # find the next sample
    x_next = x_next + epsilon * torch.sign(grad_vars[0].data)

    # projection
    x_next = clip_tensor(x_next)

In [ ]:
grad_vars[0].shape

## Step 13 — The codebase's own test entry points

In [ ]:
from core.defense import Dataset

# Dataset is a class
data = Dataset()
data.vocab_size

In [ ]:
#if you face an error in running the next line , probably because of this part of dataset
# you should remove and let the code recreate it
# probably because of diffrent pattern in the split of data to train and test and valid
#!rm -r /content/malware_detection/dataset/drebin

In [ ]:
!python -m examples.md_nn_test --cuda --cache --seed 0 --batch_size 128 --proc_number 1 --epochs 50 --max_vocab_size 10000 --dense_hidden_units "200,200" --weight_decay 0.0 --lr 0.001 --dropout 0.6

In [ ]:
!python -m examples.md_at_pgd_test  --cache --beta 1.0 --seed 0 --batch_size 128 --proc_number 10 --epochs 50 --max_vocab_size 10000 --dense_hidden_units "200,200" --weight_decay 0.0 --lr 0.001 --dropout 0.6 --steps_linf 50 --step_length_linf 0.02

In [ ]:
!python -m examples.md_at_ma_test --cuda --cache --beta 0.01 --seed 0 --batch_size 128 --proc_number 2 --epochs 50 --max_vocab_size 10000 --dense_hidden_units "200,200" --weight_decay 0.0 --lr 0.001 --dropout 0.6  --ma "max" --steps_l1 50 --steps_linf 50 --step_length_linf 0.02 --steps_l2 50 --step_length_l2 0.5